# URLvestigia — quickstart

**Run All.** Nothing to edit, nothing to install by hand, no API keys, and no
terminal at any point.

This notebook does one full pass of what the accelerator is for: it checks what
this network can reach, asks a question, keeps the answer *and the way the answer
was found* as a governed row — and then starts the dashboard and hands you a link
to it.

About a minute end to end; the preflight is the slow part. It writes one thing,
`data/urlvestigia.db`, which is gitignored, and it installs anything it needs into
this kernel as it goes.

In [ ]:
# --- bootstrap ---
# A notebook has no __file__, so the working directory is the only anchor there is.
# It is usually the notebook's own directory — but VS Code's jupyter.notebookFileRoot
# can point a kernel at the workspace folder instead, and `jupyter lab` started from
# elsewhere inherits wherever it was started. So search upward for the repository
# rather than assuming, and say so plainly when it is not there.
import sys
from pathlib import Path


def find_root(start=None):
    """The URLvestigia repository at or above `start`."""
    here = Path(start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        # Two markers, not one: scripts/new-accelerator.sh copies METADATA.yaml into
        # a fresh accelerator, so on its own it would match the wrong repository.
        if (candidate / "METADATA.yaml").is_file() and (candidate / "data" / "db.py").is_file():
            return candidate
    raise RuntimeError(
        f"No URLvestigia repository at or above {here}. "
        "Open quickstart.ipynb from inside the cloned repository.")


ROOT = find_root()
# The layers are directories, not installed packages, so they go on the path the
# same way app/server.py puts them there at runtime. ROOT itself is included so
# `from app import hosting` resolves — the same reason tests/conftest.py adds it.
for layer in ("retrieval", "data", "scripts", "."):
    path = str((ROOT / layer).resolve())
    if path not in sys.path:
        sys.path.insert(0, path)

print(f"repository  {ROOT}")
print(f"kernel      {sys.executable}")

## 1. Dependencies

The search library is one package. If this kernel does not have it, this cell
installs it — into *this* interpreter, which is frequently not the `pip` on your
PATH, and is the single most common reason a "but I installed it" notebook fails.

If the install cannot be done for you, nothing raises: the cell says what to run,
and every cell below reports itself skipped instead of throwing a traceback.

In [ ]:
from kernel import ensure

# Every layer's requirements file is installed the same way, into this kernel's own
# interpreter. scripts/kernel.py has the why.
READY = ensure(["ddgs"], ROOT / "retrieval" / "requirements.txt", "the search library")
print("ready" if READY else "not ready - see above")

## 2. What can this network actually reach?

The same preflight `make doctor` runs, probing each corpus and **each web engine
individually** — a single blocked engine is worth seeing by name rather than
hidden inside the pool.

This is the slow cell: 10–40 seconds, longer if something is timing out. A blocked
engine here is a measurement of this network, not a defect. The next cell picks a
provider from whatever answered, so a locked-down network gets a readable
diagnosis instead of a failed search three cells later.

In [ ]:
if READY:
    import doctor

    probes = doctor.rows()
    for label, status, detail, seconds in probes:
        print(f"  {doctor.MARK[status]}  {label:16} {seconds:5.1f}s  {detail}")

    healthy = [label for label, status, _, _ in probes if status == doctor.OK]
    ENGINES = [label.removeprefix("web: ") for label in healthy if label.startswith("web: ")]
    APIS = [label for label in healthy if not label.startswith("web: ")]

    # Prefer the open web when any engine answered; otherwise search a corpus this
    # network can actually reach, rather than demonstrating a failure.
    PROVIDER = "ddgs" if ENGINES else (APIS[0] if APIS else None)
    if PROVIDER is None:
        READY = False
        print("\n  Nothing is reachable from here. Check the network, a proxy, or a")
        print("  TLS-intercepting firewall; the rows above name which it is.")
    else:
        print(f"\n  Searching with: {PROVIDER}"
              + (f" via {', '.join(ENGINES)}" if ENGINES else ""))
else:
    print("skipped - see the dependency cell above")

## 3. One question, recorded

Edit `QUERY` and re-run this cell as often as you like; every run adds a row.

The part worth watching is what gets *stored*. Options this corpus does not apply
are recorded as `NULL`, not as the value passed in — so the record can never claim
a filter that never ran. That single rule is what makes the table defensible months
later, and it is enforced in one place, `data/record.py`, for the dashboard, the
terminal, and this notebook alike.

In [ ]:
QUERY = "GLP-1 receptor agonist adverse event reporting"   # <- edit me

if READY:
    import db
    import record
    import urlvestigia

    # Idempotent, and it migrates an older store in place - as app/server.py does.
    db.init_db()

    try:
        result = record.run(QUERY, provider=PROVIDER, max_results=10, backend=ENGINES)
    except urlvestigia.EngineError as exc:
        # Every engine failed and each said why; a network can drop between cells.
        print(f"{record.label(PROVIDER)} search failed - no engine answered:")
        for engine, reason in exc.failures:
            print(f"  {engine}: {reason}")
        result = None
    except Exception as exc:
        print(f"{record.label(PROVIDER)} search failed - {exc}")
        result = None

    if result and result["urls"]:
        print(f'{len(result["urls"])} urls, recorded as search #{result["search_id"]}:\n')
        for url in result["urls"]:
            print(f"  {url}")
    elif result:
        # Not an error: the corpus answered and had nothing.
        print(f'No results. {record.label(PROVIDER)} had nothing for "{QUERY}".')
        print("Try editing QUERY above - a scholarly corpus will not match a")
        print("navigational web query, and vice versa.")
else:
    print("skipped - see the cells above")

## 4. The record

The deliverable. Every row carries the question and the options that produced it,
so the search can be reproduced or audited by someone who was not here.

Read the option columns carefully — they say two different things:

* **`n/a`** — this corpus has no such option. Nothing was filtered.
* **`any`** — it has one, and this search chose not to use it.

Collapsing those two is exactly the overstatement the record exists to prevent,
which is why [`data/present.py`](data/present.py) renders the table rather than
handing the rows to pandas: a DataFrame prints both as a blank cell.

In [ ]:
import present
from IPython.display import HTML, display

if present.store():
    display(HTML(present.searches(limit=10)))
    print("n/a = this corpus has no such option.  any = it has one, unused.")
else:
    print("Nothing recorded yet - run section 3.")

## 5. Export it

One row per URL, with the provenance of its search denormalized onto it — the shape
a reviewer sorts by domain and filters by provider, and the same shape the lakehouse
curates into. `NULL` is spelled out because CSV has only one kind of empty cell and
this record needs two.

This cell only *shows* the CSV, so Run All never leaves a file behind. The command
underneath writes the real one.

In [ ]:
if READY:
    import io

    import cli

    preview = io.StringIO()
    cli.write_csv(cli.export_rows(limit=10), preview)
    for line in preview.getvalue().splitlines()[:8]:
        print(line[:160])

    print("\nTo write the file itself:")
    print('    python scripts/cli.py export --format csv --out review-appendix.csv')
else:
    print("skipped - see the cells above")

## 6. Open the dashboard

The same record, in a browser — the interface for people who will never open a
terminal. Running this cell starts the server in the background and prints a link;
the search you ran above is already in it.

Three things worth knowing.

* It **reuses a server already on the port** rather than fighting it for one, so
  this is safe to run twice, and safe if your editor already started one — but it
  asks that server who it is before trusting it. A dashboard left behind by a
  kernel you have since restarted answers a probe exactly like a fresh one while
  serving the code as it was back then, so the cell names what it found, says when
  that code was loaded, and warns you if you have edited anything since.
* It keeps running after the cell finishes — that is the point — so section 7 is
  how you stop it, a server it adopted just as much as one it started.
* **In a Cloudera AI session it binds differently.** Your browser is outside the
  container, so `127.0.0.1` would point at your own laptop and connect to nothing.
  There the server binds every interface on `CDSW_APP_PORT` and the link becomes
  the session's proxied address. On a laptop it stays on loopback — this app has no
  authentication, and putting it on the local network is the one thing
  [the README](README.md#prerequisites) tells you not to do. The rule lives in
  [`app/hosting.py`](app/hosting.py).

In [ ]:
from IPython.display import HTML, display

from app import hosting
from kernel import ensure

if READY and ensure(["fastapi", "uvicorn", "jinja2"],
                    ROOT / "app" / "requirements.txt", "the dashboard"):
    # Reuses a server already on the port, adopts one a restarted kernel left
    # behind, or starts its own - and says which. app/hosting.py decides.
    DASHBOARD = hosting.dashboard(globals().get("DASHBOARD"), ROOT)
    display(HTML(DASHBOARD.report()))
else:
    print("skipped - see the cells above")

## 7. Stop the dashboard

Left alone this does nothing, so Run All does not kill the server it just started.
Set the flag to `True` and run this cell when you are finished — or just shut the
kernel down, which stops it too.

It stops a server it adopted as readily as one it started — the leftover on the
port after a kernel restart included. Nothing holds a handle to that one, so it is
stopped by the pid `/healthz` reported, which is what keeps an orphaned server from
being something you need a terminal to clear.

In [ ]:
STOP_DASHBOARD = False   # <- set to True, then run this cell

DASHBOARD = globals().get("DASHBOARD")
print(hosting.stop(DASHBOARD) if STOP_DASHBOARD else hosting.status(DASHBOARD))

## Where to go next

Everything above ran without a terminal. These are the same capabilities for when
you want one.

**The dashboard**, started by hand instead of by section 6:

```bash
make dev                                                # → http://127.0.0.1:8000/
python -m uvicorn app.server:app --reload --port 8000   # without make
```

**The terminal**, which writes the same records and pipes:

```bash
python scripts/cli.py search "your question" --provider arxiv -n 25
python scripts/cli.py list --urls
python scripts/cli.py export --format csv --out review-appendix.csv
```

**Before a live demo:** `make doctor` — the preflight from section 2, with a verdict.

**Deeper:**

* [`docs/EXAMPLE.md`](docs/EXAMPLE.md) — one question followed through all five layers
* [`retrieval/notebooks/eval.ipynb`](retrieval/notebooks/eval.ipynb) — how the engines
  and corpora actually compare on availability and overlap
* [`docs/ARCHITECTURE.md`](docs/ARCHITECTURE.md) — the decisions worth defending